# 🏥 Geovisor - Hospitales y Clínicas ITT

**Proyecto:** Gobierno de Datos 2026 
**Módulo:** Espacialización de Instituciones Prestadoras de Salud 
**Autor:** Data Steward 
**Repositorio:** [GitHub](https://github.com/j0rg3c45/Hospitales_Cl-nicas_ITT.git)

---

## Objetivo

Generar un visor geográfico interactivo (Geovisor) que permita visualizar, filtrar y analizar individualmente cada hospital y clínica del territorio, consumiendo servicios de imágenes satelitales de Google como mapa base.

---
## 1. Configuración del Entorno e Instalación de Dependencias

In [ ]:
# ============================================================
# SECCIÓN 1: Instalación de librerías necesarias (Google Colab)
# ============================================================

!pip install geopandas folium mapclassify --quiet

print("✅ Dependencias instaladas correctamente.")

In [ ]:
# ============================================================
# SECCIÓN 1.1: Importación de librerías
# ============================================================

import json
import geopandas as gpd
import pandas as pd
import folium
from folium import plugins, GeoJson, FeatureGroup, LayerControl
from folium.plugins import MiniMap, Fullscreen, LocateControl
from google.colab import files
import os

print("✅ Librerías importadas correctamente.")
print(f"   GeoPandas: {gpd.__version__}")
print(f"   Folium: {folium.__version__}")

---
## 2. Carga de Datos Geográficos

In [ ]:
# ============================================================
# SECCIÓN 2: Carga del archivo GeoJSON desde la carpeta Data/
# ============================================================

# Opción A: Cargar desde ruta local (si se clonó el repositorio)
DATA_PATH = "../Data/hospitales_clinicas.json"

# Opción B: Subir archivo manualmente en Colab
if not os.path.exists(DATA_PATH):
    print("📂 Archivo no encontrado en ruta local.")
    print("   Suba el archivo GeoJSON manualmente:")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
    print(f"   ✅ Archivo cargado: {DATA_PATH}")

# Lectura con GeoPandas
gdf = gpd.read_file(DATA_PATH)

print(f"\n✅ Datos cargados exitosamente.")
print(f"   Registros: {len(gdf)}")
print(f"   CRS: {gdf.crs}")
print(f"   Columnas: {list(gdf.columns)}")

---
## 3. Exploración y Preprocesamiento de Datos

In [ ]:
# ============================================================
# SECCIÓN 3: Exploración inicial del GeoDataFrame
# ============================================================

# Vista previa de los datos
display(gdf.head())

# Tipos de geometría presentes
print(f"\n📐 Tipos de geometría: {gdf.geometry.geom_type.unique()}")

# Estadísticas básicas
print(f"\n📊 Resumen estadístico:")
display(gdf.describe())

In [ ]:
# ============================================================
# SECCIÓN 3.1: Validación y transformación de CRS
# ============================================================

# Asegurar que el CRS sea WGS84 (EPSG:4326) para visualización web
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)
    print("⚠️ CRS no definido. Se asignó EPSG:4326 (WGS84).")
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    print(f"🔄 CRS reproyectado a EPSG:4326 (WGS84).")
else:
    print("✅ CRS correcto: EPSG:4326 (WGS84).")

# Calcular centroides para etiquetado
gdf['centroid_lat'] = gdf.geometry.centroid.y
gdf['centroid_lon'] = gdf.geometry.centroid.x

print(f"\n📍 Extensión geográfica:")
print(f"   Lat: [{gdf['centroid_lat'].min():.4f}, {gdf['centroid_lat'].max():.4f}]")
print(f"   Lon: [{gdf['centroid_lon'].min():.4f}, {gdf['centroid_lon'].max():.4f}]")

---
## 4. 🌍 Espacialización / Geovisor

Sección principal del análisis. Se genera un mapa interactivo con:
- **Mapa base:** Imágenes satelitales de Google (Google Satellite/Maps).
- **Capas individuales:** Cada hospital/clínica como una capa independiente.
- **Control de capas (Layer Control):** Permite encender/apagar cada institución.
- **Popups informativos:** Con datos clave de cada IPS.

In [ ]:
# ============================================================
# SECCIÓN 4.1: Definición de Mapas Base (Basemaps)
# ============================================================

# Configuración de tiles de Google Maps/Satellite
GOOGLE_SATELLITE = "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}"
GOOGLE_MAPS = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
GOOGLE_HYBRID = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"

# Centro del mapa (promedio de coordenadas)
center_lat = gdf['centroid_lat'].mean()
center_lon = gdf['centroid_lon'].mean()

print(f"📍 Centro del mapa: [{center_lat:.4f}, {center_lon:.4f}]")

In [ ]:
# ============================================================
# SECCIÓN 4.2: Construcción del Geovisor Interactivo
# ============================================================

# Crear mapa base con Folium
mapa = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles=None,  # Sin tiles por defecto, se agregan manualmente
    control_scale=True
)

# --- BASEMAPS ---
# Google Satellite
folium.TileLayer(
    tiles=GOOGLE_SATELLITE,
    attr="Google Satellite",
    name="🛰️ Google Satellite",
    overlay=False
).add_to(mapa)

# Google Hybrid (Satellite + Labels)
folium.TileLayer(
    tiles=GOOGLE_HYBRID,
    attr="Google Hybrid",
    name="🗺️ Google Hybrid",
    overlay=False
).add_to(mapa)

# Google Maps (Calles)
folium.TileLayer(
    tiles=GOOGLE_MAPS,
    attr="Google Maps",
    name="🏙️ Google Maps",
    overlay=False
).add_to(mapa)

# OpenStreetMap (respaldo)
folium.TileLayer(
    tiles="OpenStreetMap",
    name="🌐 OpenStreetMap",
    overlay=False
).add_to(mapa)

print("✅ Mapas base configurados.")

In [ ]:
# ============================================================
# SECCIÓN 4.3: Adición de Capas Individuales por Institución
# ============================================================

# Definir paleta de colores para diferenciar instituciones
colores = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#469990', '#dcbeff', '#9A6324', '#fffac8', '#800000',
    '#aaffc3', '#808000', '#ffd8b1', '#000075', '#a9a9a9'
]

# Identificar columna con el nombre de la institución
# (Ajustar 'nombre' al campo real del GeoJSON)
CAMPO_NOMBRE = 'nombre'  # ← MODIFICAR según la estructura real del JSON

# Verificar que el campo existe
if CAMPO_NOMBRE not in gdf.columns:
    print(f"⚠️ Campo '{CAMPO_NOMBRE}' no encontrado.")
    print(f"   Columnas disponibles: {list(gdf.columns)}")
    print(f"   Usando el índice como identificador.")
    gdf[CAMPO_NOMBRE] = [f"Institución {i+1}" for i in range(len(gdf))]

# Iterar sobre cada institución y crear una capa independiente
for idx, row in gdf.iterrows():
    nombre_institucion = row[CAMPO_NOMBRE]
    color = colores[idx % len(colores)]

    # Crear FeatureGroup individual (permite control de visibilidad)
    fg = FeatureGroup(name=f"🏥 {nombre_institucion}", show=True)

    # Preparar popup con información
    popup_html = f"""
    <div style='font-family: Arial; font-size: 12px; min-width: 200px;'>
        <h4 style='color: {color}; margin-bottom: 5px;'>🏥 {nombre_institucion}</h4>
        <hr style='margin: 3px 0;'>
        <b>Latitud:</b> {row['centroid_lat']:.6f}<br>
        <b>Longitud:</b> {row['centroid_lon']:.6f}<br>
        <b>Tipo geometría:</b> {row.geometry.geom_type}<br>
    </div>
    """

    # Agregar geometría según tipo
    if row.geometry.geom_type in ['Polygon', 'MultiPolygon']:
        # Polígono con estilo
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, c=color: {
                'fillColor': c,
                'color': c,
                'weight': 2,
                'fillOpacity': 0.4
            },
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=nombre_institucion
        ).add_to(fg)
    else:
        # Punto con marcador
        folium.Marker(
            location=[row['centroid_lat'], row['centroid_lon']],
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=nombre_institucion,
            icon=folium.Icon(color='red', icon='plus-sign', prefix='glyphicon')
        ).add_to(fg)

    # Añadir FeatureGroup al mapa
    fg.add_to(mapa)

print(f"✅ {len(gdf)} instituciones agregadas como capas individuales.")

In [ ]:
# ============================================================
# SECCIÓN 4.4: Controles del Mapa y Layer Control
# ============================================================

# Control de capas (permite encender/apagar cada institución)
LayerControl(
    collapsed=False,
    position='topright'
).add_to(mapa)

# Minimapa de referencia
MiniMap(
    toggle_display=True,
    position='bottomleft'
).add_to(mapa)

# Botón de pantalla completa
Fullscreen(
    position='topleft',
    title='Pantalla completa',
    title_cancel='Salir de pantalla completa'
).add_to(mapa)

# Control de ubicación del usuario
LocateControl(
    strings={'title': 'Mi ubicación'}
).add_to(mapa)

print("✅ Controles del mapa configurados.")
print("   - Layer Control (encender/apagar capas)")
print("   - MiniMap")
print("   - Fullscreen")
print("   - Locate Control")

---
## 5. Visualización del Geovisor

In [ ]:
# ============================================================
# SECCIÓN 5: Renderizar el mapa interactivo en el notebook
# ============================================================

# Mostrar el Geovisor
mapa

---
## 6. Exportación de Resultados

In [ ]:
# ============================================================
# SECCIÓN 6: Exportar el mapa como archivo HTML interactivo
# ============================================================

OUTPUT_PATH = "../Outputs/geovisor_hospitales_clinicas_ITT.html"

# Guardar mapa
mapa.save(OUTPUT_PATH)
print(f"✅ Mapa exportado exitosamente: {OUTPUT_PATH}")

# Descargar en Colab (opcional)
try:
    files.download(OUTPUT_PATH)
    print("📥 Descarga iniciada.")
except:
    print("ℹ️ Para descargar manualmente, navegue a la carpeta Outputs/.")

---
## 7. Conclusiones y Próximos Pasos

### Resultados obtenidos:
- Geovisor interactivo con mapa base de Google Satellite.
- Cada institución representada como capa individual con control on/off.
- Popups informativos con datos clave de cada IPS.

### Próximos pasos:
- Enriquecer los popups con datos de capacidad, servicios y contacto.
- Incorporar análisis de cobertura y áreas de influencia.
- Integrar indicadores de calidad de datos (completitud, exactitud).
- Generar reportes automáticos por zona geográfica.